
# 03 · Reinforcement Learning — DDPG + PPO

**Vai trò notebook**: kiến trúc DDPG (paper Xiong et al replica) + PPO backup, training curves, sample decisions, lessons learned (DDPG saturated tanh → PPO khắc phục).

**Owner**: Duc
**Deadline**: 2026-05-22
**Slide chapter**: 4 — Implementation (Strategies → RL subsection)

## Mục tiêu
1. Kiến trúc actor-critic của DDPG (continuous action) + PPO (clipped objective)
2. Training log → curves (actor_loss, critic_loss, episode_return)
3. Sample forward pass: state → action
4. Failure analysis: DDPG saturated tanh (PRD §14 Risk #7) + bằng chứng

## Defense Q&A
- Q: DDPG khác PPO ở điểm gì? Tại sao PPO khắc phục được vấn đề DDPG?
- Q: Network architecture chi tiết (layers, hidden_dim, activation)?
- Q: State + action space định nghĩa thế nào?
- Q: Reward function?


## Setup


In [ ]:
import sys
from pathlib import Path

_NB_DIR = Path().resolve()
if _NB_DIR.name != "notebooks":
    _NB_DIR = _NB_DIR / "notebooks"
if str(_NB_DIR) not in sys.path:
    sys.path.insert(0, str(_NB_DIR))

from _shared import (  # noqa: E402
    AGENT_COLORS,
    BASELINES,
    DATA,
    FIGURES,
    LLM_AGENTS,
    RESULTS,
    RL_AGENTS,
    ROLE_COLORS,
    TRANSCRIPTS,
    assert_frozen_snapshot,
    list_transcript_dates,
    load_curve,
    load_holdings,
    load_metrics_json,
    load_metrics_table,
    load_transcript,
    save_fig,
    setup_matplotlib,
)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

setup_matplotlib()
assert_frozen_snapshot()
metrics = load_metrics_table()
print("snapshot OK · agents:", list(metrics.index))



## TODO-01: state-action-space — formalize MDP
- **OWNER**: Duc   **DEPENDS**: none
- **READ**: `src/trading_env.py:VNTradingEnv.observation_space`, `action_space`, `_reward`
- **WRITE**: markdown cells với spec MDP + code cell in ra space shapes
- **CONSTRAINTS**:
  Markdown sections:
  ```
  ### State (s_t)
  - Per-ticker (×5): price OHLCV window, RSI, MACD, Bollinger
  - Portfolio: current weights, cash ratio
  - Total dim: <X>
  ### Action (a_t)
  - Continuous target weights ∈ [-1, 1]^5
  - Long-only? Long-short?
  ### Reward (r_t)
  - r_t = log(value_t / value_{t-1}) - λ * turnover  (or similar)
  ### Discount γ
  - 0.99 (config)
  ```
- **VALIDATE**:
  ```python
  env = VNTradingEnv(...)
  print(env.observation_space.shape, env.action_space.shape)
  ```
- **DEFENSE Q&A**: "State / action / reward của bạn?" → đọc 3 sections này.


In [ ]:
# TODO-01: print env spaces
from src.trading_env import VNTradingEnv
from src.env_data_loader import load_market_data
md = load_market_data('train')
env = VNTradingEnv(md)
print('obs:', env.observation_space)
print('act:', env.action_space)



## TODO-02: ddpg-arch — actor-critic networks
- **OWNER**: Duc   **DEPENDS**: TODO-01
- **READ**: `src/ddpg_trainer.py`, stable-baselines3 DDPG default + custom
- **WRITE**:
  - Markdown: actor (deterministic μ(s)) + critic (Q(s,a)), OU noise
  - `report/figures/03__ddpg_arch.png` — diagram actor + critic + replay buffer + target nets
  - Code cell: load `results/models/ddpg_best.zip`, in `policy` summary
- **CONSTRAINTS**:
  - Hidden dims từ trainer config
  - Note OU noise std σ giảm trong training
- **VALIDATE**: model loads; `policy.actor` + `policy.critic` đều có
- **PATTERN**: `stable_baselines3.DDPG.load(path).policy`


In [ ]:
# TODO-02: DDPG arch + load model
from stable_baselines3 import DDPG
model = DDPG.load(RESULTS / 'models' / 'ddpg_best.zip')
print(model.policy)



## TODO-03: ppo-arch — clipped objective + entropy bonus
- **OWNER**: Duc   **DEPENDS**: TODO-02
- **READ**: `src/ppo_trainer.py`
- **WRITE**: markdown + `report/figures/03__ppo_arch.png`
- **CONSTRAINTS**:
  - PPO clipped surrogate: L^CLIP(θ) = E[ min(r_t(θ) A_t, clip(r_t, 1-ε, 1+ε) A_t) ]
  - ε = 0.2 (default sb3)
  - Stochastic policy (Gaussian over actions) — KEY DIFFERENCE vs DDPG deterministic
- **PATTERN**: `stable_baselines3.PPO.load(path).policy`


In [ ]:
# TODO-03: PPO arch
from stable_baselines3 import PPO
model = PPO.load(RESULTS / 'models' / 'ppo_best.zip')
print(model.policy)



## TODO-04: training-curves — DDPG vs PPO learning
- **OWNER**: Duc   **DEPENDS**: TODO-02, 03
- **READ**: `results/ddpg_training_log.jsonl`, `results/ppo_training_log.jsonl`
- **WRITE**: `report/figures/03__training_curves.png`
- **CONSTRAINTS**:
  - 3-panel: (a) episode_return, (b) actor_loss, (c) critic_loss / value_loss
  - 2 lines mỗi panel (DDPG vs PPO)
  - X = training step
  - PPO converge stable, DDPG actor_loss → 0 sớm (saturated) → explanation
- **VALIDATE**: file exists; PPO final return > DDPG final return
- **DEFENSE Q&A**: "Training stable không?" → mở figure này


In [ ]:
# TODO-04: training curves
ddpg_log = [json.loads(l) for l in open(RESULTS / 'ddpg_training_log.jsonl')]
ppo_log = [json.loads(l) for l in open(RESULTS / 'ppo_training_log.jsonl')]
# ...



## TODO-05: ddpg-saturated-tanh — root-cause analysis
- **OWNER**: Duc   **DEPENDS**: TODO-04
- **READ**: `results/ddpg/holdings.parquet`, `results/ddpg/portfolio_curve.parquet`
- **WRITE**: markdown analysis + `report/figures/03__ddpg_saturated.png`
- **CONSTRAINTS**:
  - Time-series plot: 5 weight columns vs date trong test window
  - Show: DDPG overweight HPG (≈ +100%) gần như mọi ngày → saturated boundary
  - Markdown explain: tanh ≈ ±1 → gradient ≈ 0 → policy stuck
- **VALIDATE**:
  ```python
  h = load_holdings('ddpg')
  hpg_concentration = h['HPG'] / h[['VCB','FPT','HPG','VIC','VNM']].sum(axis=1)
  assert hpg_concentration.mean() > 0.5  # > 50% concentration
  ```
- **DEFENSE Q&A** (critical): "DDPG return chỉ +1% — explain?" → saturated tanh + concentration risk; chuyển sang PPO.


In [ ]:
# TODO-05: DDPG saturated tanh evidence
pass



## TODO-06: sample-decisions — 1 forward pass mỗi agent
- **OWNER**: Duc   **DEPENDS**: TODO-02, 03
- **WRITE**: 2 code cells, mỗi cái show state → action cho 1 timestep
- **CONSTRAINTS**:
  ```python
  obs = env.reset()
  action_ddpg, _ = ddpg_model.predict(obs, deterministic=True)
  action_ppo, _ = ppo_model.predict(obs, deterministic=True)
  print("DDPG:", dict(zip(TICKERS, action_ddpg)))
  print("PPO:", dict(zip(TICKERS, action_ppo)))
  ```
- **VALIDATE**: actions ∈ [-1, 1]^5
- **DEFENSE Q&A**: "Cho xem 1 quyết định cụ thể của RL?" → 2 cell này


In [ ]:
# TODO-06: sample DDPG + PPO actions
pass



## Defense Q&A — câu trả lời sẵn

> **Q1: DDPG vs PPO khác gì?**
> A: DDPG deterministic policy + Q-learning + OU noise → exploration weak khi action saturate. PPO stochastic Gaussian policy + clipped surrogate + entropy bonus → robust exploration, no boundary stuck.
> Evidence: TODO-04 training curves + TODO-05 DDPG saturation proof.

> **Q2: Network architecture?**
> A: Cả 2 actor + critic là MLP. Hidden dims từ `ddpg_trainer.py` / `ppo_trainer.py` config (đọc cell TODO-02, 03 để in chi tiết).

> **Q3: State / action / reward?**
> A: State = window features (TODO-01). Action ∈ [-1,1]^5 = target weights. Reward = log return - λ × turnover (TODO-01).

> **Q4: Tại sao không SAC/TD3?**
> A: Out of scope (PRD §4). DDPG là replica từ paper Xiong et al; PPO là backup. SAC/TD3 explore là post-MVP work.
